# 실습 1. 주제분류 — BERT로 풀고, T5로 다시 풀기

**AI아카데미 [A4021] 언어지능: 언어모델 기반 자연어처리 실습 기초 · 1일차 오후**

뉴스 제목 하나를 읽고 **7가지 주제 중 하나**를 고릅니다.

제목: `유튜브 내달 2일까지 크리에이터 지원 공간 운영`  →  정답: `IT과학`

라벨은 일곱 개입니다 — `IT과학`, `경제`, `사회`, `생활문화`, `세계`, `스포츠`, `정치`.

## 오늘 하는 일

같은 문제를 **두 가지 구조**로 풉니다. 데이터도, 평가셋도, 채점 함수도 완전히 같습니다.
그래야 "구조가 다르면 무엇이 달라지는가"를 숫자로 볼 수 있습니다.

| | **BERT** (인코더) | **pko-T5** (인코더-디코더) |
|---|---|---|
| 하는 일 | 읽고 **고른다** — 라벨·태그·위치 | 읽고 **써낸다** — 답을 글자로 |
| 태스크를 바꾸려면 | head(작은 출력층)와 손실 함수를 바꾼다 | 지시문(앞에 붙이는 말)을 바꾼다 |
| 없는 말을 지어낼 수 있나 | **못 한다** (구조상 불가능) | 할 수 있다 |
| 이 노트북에서 | **여러분이 직접 만듭니다** | 강사가 시연하고, 그대로 실행합니다 |

내일(2일차)은 세 번째 구조인 **GPT 계열(디코더)** 입니다. 마지막 비교표에 미리 넣어 두었습니다.

## 순서와 소요시간

| | 내용 | 시간 |
|---|---|---|
| §1~3 | 환경 확인 · 데이터 보기 · 토크나이저 살펴보기 | 10분 |
| §4 | **미션 세 개** — 분류 head 만들기와 라벨 붙이기 · 출력 되돌리기 (객관식으로 고르고, 빈칸 세 곳 채우기 — 셀 두 개) | 20분 |
| §5~7 | BERT 학습 · 평가 · 맞힌 예·틀린 예 보기 | 15분 |
| §8 | T5 시연 — 같은 문제를 '써내는' 방식으로 | 25분 |
| §9~11 | 비교 · 퀴즈 · 더 해보기 | 10분 |

> **진행 방식.** 셀을 **위에서부터 하나씩** `Shift+Enter`로 실행합니다.
> 미션은 **객관식 카드**로 나옵니다 — 보기를 고르면 맞았는지 바로 알려 주고, 틀리면 힌트가 하나씩 열립니다.
> 고른 답을 아래 미션 셀의 `____` 자리에 옮겨 적으세요. 빈칸을 채우지 않으면 확인 셀에서
> `NameError: name '____' is not defined` 로 멈춥니다.
> 이 노트북이 부르는 함수는 전부 `day1/lab_common.py`에 있고, 그 원본은
> `task5-llm-ft/bert_baseline.py`·`t5_baseline.py`입니다. 터미널에서도 똑같이 재현할 수 있습니다.

## 1. 환경 확인

노트북이 어느 폴더에 있든 **저장소 루트**를 작업 폴더로 삼습니다. 데이터는 `data/`, 결과는 `output/day1/`에 둡니다.

In [ ]:
import os, sys, json, time
from pathlib import Path

# 노트북이 day1/ 또는 day1/instructor/ 에 있어도 저장소 루트를 찾아 이동한다
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "day1" / "lab_common.py").exists()), None)
assert _root is not None, "DeepKNLP 저장소 안에서 이 노트북을 여세요"
os.chdir(_root); sys.path.insert(0, str(_root / "day1"))

import lab_common as L
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoModelForTokenClassification)
from common import TASKS, TC_LABELS, NER_LABELS

TASK = "tc"
MODEL = L.MODEL_BERT            # klue/roberta-base — 한국어 인코더
MAX_LEN = L.MAX_LEN             # 128 토큰
EPOCHS, LR, BATCH = 3, 5e-5, 32                 # BERT 학습 설정
T5_EPOCHS, T5_LR, T5_BATCH = 3, 3e-4, 8         # T5 학습 설정

L.env_info()

## 2. 데이터 보기

학습에 쓰는 것은 **800건**입니다. 적어 보이지만 일부러 이 크기로 맞췄습니다 —
2일차에 LLM을 학습시킬 때 쓴 것과 **똑같은 예제들**이라, 나중에 세 방식을 나란히 놓고 비교할 수 있습니다.

평가는 따로 떼어 둔 **300건**으로 합니다. 이 300건은 **학습에 절대 쓰지 않습니다.**

In [ ]:
train_rows = L.load_budget(TASK)          # 학습셋 (LLM·T5가 학습한 그 예제들)
eval_rows = L.load_eval(TASK)             # 평가셋 300건 — 학습에는 쓰지 않는다
L.check_no_leak(train_rows, eval_rows)

print("\n[학습셋 예시 3건]")
L.preview(train_rows, 3)

In [ ]:
print("[평가셋 예시 3건]")
L.preview(eval_rows, 3)

## 3. 토크나이저 — 모델은 글자가 아니라 '조각'을 본다

모델은 문장을 그대로 읽지 않습니다. 먼저 **서브워드**라는 조각으로 쪼갠 뒤, 조각마다 번호(id)를 붙여 넣습니다.
한국어는 조사·어미가 붙어 단어가 길어지기 때문에, 이 조각이 단어와 일치하지 않는 경우가 많습니다.
이 사실은 특히 **개체명인식**에서 중요해집니다 — 정답은 글자 단위인데 모델이 보는 단위는 조각이기 때문입니다.

In [ ]:
tokenizer = L.load_bert_tokenizer(MODEL)
L.show_tokens(tokenizer, train_rows[0]["input"]["text"])

## 4. 미션 — 분류 head 만들기와 라벨 붙이기

사전학습된 BERT는 "문장을 이해하는 몸통"까지만 갖고 있습니다. **무엇을 답할지는 아직 모릅니다.**
그래서 몸통 위에 태스크에 맞는 작은 출력층(**head**)을 새로 얹고, 그 부분까지 함께 학습시킵니다.

주제분류의 head는 문장 하나 → 라벨 하나. 사전학습된 인코더 위에 **출력이 7개인 선형층**(분류 head)을 새로 얹습니다.

아래 **객관식**을 풀고, 고른 답을 미션 셀의 빈칸에 옮겨 적으면 됩니다.

정할 것은 셋입니다. 앞의 둘은 **모델에 넣는 쪽**, 마지막 하나는 **모델이 낸 것을 받는 쪽**입니다.

| 문항 | 정할 것 | 어디 |
|---|---|---|
| `m-tc-1` | head의 **출력 개수** | 미션 셀 ① |
| `m-tc-2` | 정답 라벨을 **어떤 형태**로 넣을지 | 미션 셀 ① |
| `m-tc-3` | 모델이 낸 **번호를 무엇으로 되돌릴지** | 미션 셀 ② |

In [ ]:
L.quiz("m-tc-1")

In [ ]:
L.quiz("m-tc-2")

### 빈칸 채우기

위에서 고른 답을 아래 `____` 두 곳에 옮겨 적고 셀을 실행하세요. **나머지는 그대로 둡니다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  미션 m-tc-1 · m-tc-2 — 분류 head 만들기와 라벨 붙이기
# ══════════════════════════════════════════════════════════════════════
# 아래 코드에서 ____ **두 곳만** 채우세요. 나머지는 그대로 두면 됩니다.
# 바로 위 두 카드에서 고른 답을 그대로 옮겨 적으면 됩니다.
#
# 완성본은 `task5-llm-ft/bert_baseline.py` 에도 있습니다 — 감추지 않고 알려 드립니다.
# 먼저 스스로 골라 보고, 막히면 위 카드의 힌트를 하나씩 열어 보세요. 그래야 남는 것이 있습니다.

def build_model():
    # 사전학습된 인코더 위에 '라벨 수만큼 출력이 있는 선형층'(분류 head)을 새로 얹는다.
    # 이 층은 무작위로 시작하므로 "일부 가중치가 초기화되지 않았다"는 경고가 나오는 것이 정상이다.
    return AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=____)   # m-tc-1


def encode_choice(tokenizer, rows, max_len=MAX_LEN):
    enc = tokenizer([r["input"]["text"] for r in rows], truncation=True, max_length=max_len)
    enc["labels"] = [____ for r in rows]   # m-tc-2
    return Dataset.from_dict(dict(enc))

### 확인

빈칸을 제대로 채웠는지 작은 검사로 점검합니다. **`통과`가 찍히면** 아래로 계속 진행하세요.

- `NameError: name '____' is not defined` — 빈칸이 아직 남아 있습니다. 위 미션 셀을 보세요.
- `AssertionError` — 채운 값이 틀렸습니다. 메시지에 무엇이 어긋났는지 적혀 있습니다.

In [ ]:
_ds = encode_choice(tokenizer, train_rows[:8])
assert len(_ds) == 8, f"예제 8건을 넣었는데 결과가 {len(_ds)}건입니다"
assert "input_ids" in _ds.column_names, "토큰화 결과(input_ids)가 없습니다"
assert "labels" in _ds.column_names, "정답 라벨(labels)이 없습니다"
assert all(isinstance(v, int) for v in _ds["labels"]), "labels 는 정수여야 합니다"
assert all(0 <= v < len(TC_LABELS) for v in _ds["labels"]), f"labels 는 0~{len(TC_LABELS) - 1} 범위여야 합니다"

_m = build_model()
assert _m.config.num_labels == len(TC_LABELS), \
    f"num_labels 가 {len(TC_LABELS)} 이어야 합니다 (지금 {_m.config.num_labels})"
del _m
L.free_gpu()

print("통과 — 앞 8건의 라벨:", _ds["labels"])
print("        사람 말로   :", [TC_LABELS[v] for v in _ds["labels"]])

### 미션 셀 ② — `m-tc-3`

여기까지가 **모델에 넣는 쪽**이었습니다. 이제 반대쪽입니다 — **모델이 낸 것을 받는 쪽**.

이 과정의 목표는 BERT · T5 · LLM 셋을 **같은 자로 재는 것**입니다.
그런데 셋의 출력이 생김새부터 다릅니다.

| | 무엇을 내놓나 |
|---|---|
| BERT | 라벨마다(또는 토큰마다) **점수**. 숫자입니다. |
| T5 · LLM | 답을 **글자로 써냅니다**. 처음부터 문자열입니다. |

그래서 BERT 쪽 숫자를 **LLM 이 내놓는 것과 같은 형식의 문자열**로 되돌립니다.
그러고 나면 셋을 **하나의 채점 함수**에 그대로 넣을 수 있습니다 — 채점을 세 벌 만들 필요가 없습니다.

In [ ]:
L.quiz("m-tc-3")

위에서 고른 답을 아래 `____` **한 곳**에 옮겨 적고 셀을 실행하세요.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  미션 m-tc-3 — 라벨 번호를 무엇으로 되돌릴까
# ══════════════════════════════════════════════════════════════════════
# ____ **한 곳만** 채우세요. 위 카드에서 고른 답을 그대로 옮겨 적으면 됩니다.
#
# 이 함수가 하는 일: 모델이 낸 **숫자**를 LLM·T5 와 같은 형식의 **문자열**로 되돌린다.
# 셋을 같은 채점 함수로 재려면 출력 형식이 같아야 하기 때문입니다.

def to_pred_strings(logits):
    """BERT 가 낸 점수를 LLM·T5 와 **같은 형식의 문자열**로 되돌린다.

    BERT 는 라벨마다 점수를 낸다 — 7개짜리 점수 줄 하나가 예제 하나의 답이다.
    가장 높은 자리를 고르면 **라벨 번호**(0~6)가 나오는데, 채점 함수가 기다리는 것은
    LLM 이 써내는 것과 같은 **라벨 이름**('경제', '스포츠' …)이다. 번호를 이름으로 되돌린다.
    """
    best = logits.argmax(-1)                 # 예제마다 점수가 가장 높은 자리 = 라벨 번호
    return [____[int(i)] for i in best]   # m-tc-3

### 확인

이 함수가 제대로 도는지 **작은 가짜 입력**으로 점검합니다. 모델을 돌리지 않으므로 즉시 끝납니다.

In [ ]:
import numpy as _np
_fake = _np.zeros((3, len(TC_LABELS)));  _fake[0, 1] = 9;  _fake[1, 5] = 9;  _fake[2, 0] = 9
_got = to_pred_strings(_fake)
assert _got == ["경제", "스포츠", "IT과학"], f"기대: ['경제','스포츠','IT과학'] / 받은 값: {_got}"
assert all(isinstance(x, str) for x in _got), "채점 함수는 문자열을 받습니다 — 숫자가 섞여 있습니다"
print("통과 — 라벨 번호가 이름으로 되돌아옵니다:", _got)

## 5. BERT 학습

설정은 `task5-llm-ft/bert_baseline.py --mode budget`과 같습니다 — epochs 3, 학습률 5e-5, 배치 32, 최대 길이 128.
체크포인트는 남기지 않습니다(`save_strategy="no"`). 강의장 GPU에서 **10초 안팎**이면 끝납니다.

학습 중 나오는 `loss` 값이 내려가면 모델이 배우고 있다는 뜻입니다.
같은 코드가 같은 점수를 내도록 학습 직전에 시드를 고정합니다(`L.set_seed(42)`).

In [ ]:
L.set_seed(42)              # 새로 얹는 head의 무작위 초기화를 고정 — 같은 코드는 같은 점수를 낸다

train_ds = encode_choice(tokenizer, train_rows)
eval_ds = encode_choice(tokenizer, eval_rows)
model = build_model()
print(f"학습 {len(train_ds)}건 / 평가 {len(eval_ds)}건")

trainer, bert_meta = L.train_bert(model, tokenizer, train_ds, TASK,
                                  epochs=EPOCHS, lr=LR, batch_size=BATCH)

### 퀴즈 `q-tc-3` — 평가셋 300건으로 중간 점검을 하면

방금 화면에 찍힌 「학습 720건 · 검증 80건」이 왜 그렇게 나뉘는지 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-tc-3")

## 6. BERT 평가

평가셋 300건을 예측한 뒤, **LLM·T5와 똑같은 채점 함수**(`common.score_task`)로 채점합니다.
그러기 위해 BERT의 예측(라벨 번호·실수·토큰 태그)을 먼저 **문자열**로 바꿉니다 —
LLM이 내놓는 형식과 같게 맞춰야 같은 자로 잴 수 있기 때문입니다.

이 태스크의 대표 지표는 **정확도** 입니다.

In [ ]:
# 미션 셀 ②에서 만든 to_pred_strings 가 여기서 쓰입니다
logits = L.raw_logits(trainer, eval_ds)
preds = to_pred_strings(logits)
print("예측 예시 3건:", preds[:3], "\n")

bert_summary = L.evaluate(TASK, preds, eval_rows)
L.save_result(TASK, "bert", bert_summary, bert_meta)

### 퀴즈 `q-tc-2` — 왜 주제분류에만 macro-F1 인가

방금 찍힌 정확도 옆에 macro-F1 이 함께 나온 이유를 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-tc-2")

## 7. 맞힌 예와 틀린 예 보기

점수 하나만 보면 모델이 **무엇을 해내고 어디서 무너지는지** 알 수 없습니다.
**가장 잘 맞힌 세 건을 먼저, 가장 크게 틀린 세 건을 그다음에** 봅니다.

틀린 것만 보면 "이 모델은 못 쓰겠다"로 읽히기 쉽지만, 실제로는 대부분을 맞히고 있습니다.
둘을 나란히 놓고 보세요 — **맞힌 것과 틀린 것이 어떻게 다른 문제인지**가 이 절의 질문입니다.
사람이 봐도 헷갈리는 것인지, 아니면 모델이 못 배운 종류인지 옆 사람과 이야기해 보세요.

In [ ]:
L.examples(TASK, preds, eval_rows, 3)      # 맞힌 것 3건 + 틀린 것 3건

## 8. T5 — 같은 문제를 '써내는' 방식으로

> **여기부터는 강사 시연입니다.** 학생은 셀을 그대로 실행하면서 화면을 보세요.

지금까지 BERT는 정해진 후보 중에서 **골랐습니다**. `pko-t5-base`는 다릅니다 —
지시문을 읽고 답을 **글자로 써냅니다**. 그래서 head를 바꿀 필요가 없습니다.
바뀌는 것은 모델이 아니라 **입력에 적어 주는 지시문**입니다.

여기서 쓰는 지시문과 정답 형식은 2일차에 LLM에게 줄 것과 **한 글자도 다르지 않습니다**
(`task5-llm-ft/common.py`의 `TASKS`에 한 곳에 모아 두었습니다).
같은 문장을 주어야 세 방식을 공정하게 비교할 수 있습니다.

In [ ]:
# BERT가 쓰던 GPU 메모리를 돌려준다 (나중에 다시 돌아와 이 셀만 실행해도 되게, 변수가 없으면 조용히 넘어간다)
for _n in ("model", "trainer"):
    globals().pop(_n, None)
L.free_gpu()

t5_tok, t5_model = L.load_t5()
_src, _tgt = L.to_text(train_rows[:1])
print("─ T5가 받는 입력 ──────────────────────────────")
print(_src[0])
print("─ T5가 써내야 하는 답 ─────────────────────────")
print(_tgt[0])
print("───────────────────────────────────────────────\n")

L.set_seed(42)
t5_train_ds = L.encode_t5(t5_tok, train_rows, TASK)
t5_trainer, t5_meta = L.train_t5(t5_model, t5_tok, t5_train_ds, TASK,
                                 epochs=T5_EPOCHS, lr=T5_LR, batch_size=T5_BATCH)

### T5 평가

BERT와 **똑같은 평가셋 300건**을 씁니다. 다만 예측을 만드는 방식이 다릅니다 —
BERT는 점수가 가장 높은 것을 한 번에 고르지만, T5는 글자를 하나씩 이어 붙여 답을 완성합니다(생성).
그래서 시간이 더 걸립니다.

In [ ]:
t5_preds = L.generate_t5(t5_model, t5_tok, eval_rows, TASK)
print("생성 예시 3건:", t5_preds[:3], "\n")

t5_summary = L.evaluate(TASK, t5_preds, eval_rows)
L.save_result(TASK, "t5", t5_summary, t5_meta)

## 9. 비교 — 같은 데이터, 같은 평가셋, 같은 채점

두 방식의 점수를 나란히 놓습니다. **어느 쪽이 이겼는지보다, 왜 그런지**가 중요합니다.

- BERT는 후보 밖의 답을 낼 수 없습니다. 그래서 형식이 깨지는 일이 없습니다.
- T5는 무엇이든 써낼 수 있습니다. 형식을 스스로 지켜야 하고, 못 지키면 그만큼 점수를 잃습니다
  (`형식깨짐` 건수를 보세요).
- 학습 데이터가 800건뿐이라는 점도 함께 생각해 보세요. 데이터를 늘리면 어느 쪽이 더 이득을 볼까요?

In [ ]:
L.compare(TASK, bert_summary, t5_summary)

## 10. 퀴즈

실습에서 손으로 해 본 것을 **왜 그렇게 하는지**로 되짚는 문항입니다.
보기를 고르면 맞았는지 바로 알려 주고, 틀리면 힌트가 하나씩 열립니다.
바로 답을 보는 버튼도 있지만, **힌트를 먼저 여는 쪽**이 남는 것이 많습니다.

In [ ]:
L.quiz("q-tc")

## 11. 더 해보기

시간이 남으면 하나만 골라서 해 보세요. 답을 맞히는 것이 목적이 아니라, **무엇이 얼마나 달라지는지** 보는 것이 목적입니다.

1. **데이터를 늘리면?** 지금은 800건으로 배웠습니다. 공식 학습셋 전체로 바꾸면 어떻게 될까요.
   터미널에서 (A6000 기준 약 11.7분):

   ```
   python task5-llm-ft/bert_baseline.py --task tc --mode full --save output/day1/tc-bert-full.json
   ```

   양이 적으면 `--max-train 20000` 처럼 상한을 두세요.

2. **다른 인코더로 바꾸면?** `MODEL`을 `beomi/kcbert-base`나 `monologg/koelectra-base-v3-discriminator`로
   바꿔서 §5(BERT 학습)부터 다시 실행해 보세요. 토크나이저가 다르면 같은 문장도 다르게 쪼개집니다.

3. **시드를 바꾸면?** §5(BERT 학습)의 `L.set_seed(42)`를 `L.set_seed(7)`로 바꿔 다시 실행해 보세요.
   같은 코드인데 점수가 얼마나 달라지나요? 학습 데이터가 적을수록 이 차이가 커집니다.
   (문장유사도는 특히 크게 흔들립니다 — 그래서 그 노트북만 epochs를 5로 두었습니다.)

4. **T5의 형식 깨짐.** T5 예측 중 형식이 어긋난 것을 직접 찾아보세요 — `t5_preds`를 훑어보면 됩니다.
   BERT에서는 왜 이런 일이 생길 수 없는지 설명할 수 있나요?

---

내일은 세 번째 구조인 **GPT 계열(디코더)** 입니다. 오늘 만든 세 태스크에 기계독해·SQL생성·수학추론을 더해
**여섯 태스크를 모델 하나로** 배웁니다.